# Hierarchical fusion with fine-tuned ResNet-50 V2 features

Padding is excluded from the Bi-LSTM, text guidance, and text reconstruction attention.

This follows the paper’s three-modality architecture: ResNet-50 V2 regional image vectors, five image attributes predicted by your COCO-trained ResNet-101 (loaded from `ATTRIBUTE_PREDICTIONS`), and GloVe text vectors fed to a Bi-LSTM initialized by attribute guidance; representation fusion and modality fusion feed the classifier. The fusion layers and Bi-LSTM are trained here.

This is a pipeline-level reproduction, not an exact reproduction of every experiment setting: ResNet-50 fine-tuning, COCO caption-label construction, and training schedules use explicit implementation choices. Set `FEATURE_DIR` to the mounted folder of `.npy` features.


In [1]:
import ast, csv, json, os, pickle, platform, random, time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
FEATURE_DIR = Path('/kaggle/input/datasets/fruich/mmsd-full/data/features/resnetv2_50_a1h_in1k_finetuned')
ATTRIBUTE_PREDICTIONS = Path('/kaggle/input/datasets/fruich/resnet101-coco/resnet101_coco_attributes/extract_all')
SEEDS, BATCH_SIZE, EPOCHS, PATIENCE = (42, 43, 44), 32, 8, 5
LEARNING_RATE, L2, SEQ_LEN, NUM_WORKERS = 0.001, 1e-7, 75, 4
RUN_ROOT = Path('/kaggle/working/runs/hierarchical_finetuned_unfiltered_packed')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


PyTorch: 2.11.0+cu130 Device: cuda


## Text, attribute, and image features

In [2]:
class VocabularyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global references are not allowed in vocabularies')

def load_vocabulary(path):
    with path.open('rb') as stream:
        return VocabularyLoader(stream, encoding='latin1').load()

word_vocab = load_vocabulary(ASSET_ROOT / 'words/vocab')
attribute_vocab = load_vocabulary(ASSET_ROOT / 'ExtractWords/vocab')
word_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'words/vector', dtype=np.float32))
attribute_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'ExtractWords/vector', dtype=np.float32))
assert word_vocab['<pad>'] == 0
attributes = {str(row[0]): row[1:] for row in
              (ast.literal_eval(line) for line in ATTRIBUTE_PREDICTIONS.read_text().splitlines() if line.strip())}
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}

class SarcasmDataset(Dataset):
    def __init__(self, split):
        rows = [ast.literal_eval(line) for line in
                (ASSET_ROOT / f'text/{split}.txt').read_text().splitlines() if line.strip()]
        self.rows = [row for row in rows if not blocked.intersection(row[1].split())]
        self.tokens = torch.zeros((len(self.rows), SEQ_LEN), dtype=torch.long)
        self.attributes = torch.empty((len(self.rows), 5), dtype=torch.long)
        self.labels = torch.tensor([row[-1] for row in self.rows], dtype=torch.float32)
        for i, row in enumerate(self.rows):
            token_ids = [word_vocab.get(token, word_vocab['<unk>']) for token in row[1].split()[:SEQ_LEN]]
            self.tokens[i, :len(token_ids)] = torch.tensor(token_ids, dtype=torch.long)
            self.attributes[i] = torch.tensor(
                [attribute_vocab.get(token, attribute_vocab['<unk>']) for token in attributes[str(row[0])]],
                dtype=torch.long)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        tweet_id = str(self.rows[index][0])
        image = torch.from_numpy(np.load(FEATURE_DIR / f'{tweet_id}.npy', allow_pickle=False))
        return self.tokens[index], image, self.attributes[index], self.labels[index], tweet_id

train_data, valid_data, test_data = (SarcasmDataset(split) for split in ('train', 'valid2', 'test2'))
generator = torch.Generator().manual_seed(SEEDS[0])
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', generator=generator,
                          persistent_workers=NUM_WORKERS > 0)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                         pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
positive = train_data.labels.sum().item()
positive_weight = (len(train_data) - positive) / positive
print('Train / validation / test:', len(train_data), len(valid_data), len(test_data))


Train / validation / test: 19816 2410 2409


## Hierarchical fusion model

In [3]:
class GuidedAttention(nn.Module):
    def __init__(self, raw_size, guidance_size):
        super().__init__()
        hidden_size = (raw_size + guidance_size) // 2
        self.raw = nn.Linear(raw_size, hidden_size)
        self.guidance = nn.Linear(guidance_size, hidden_size, bias=False)
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, raw, guidance, mask=None):
        hidden = torch.tanh(self.raw(raw) + self.guidance(guidance).unsqueeze(1))
        scores = self.score(hidden)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        return scores.softmax(dim=1)

class HierarchicalFusion(nn.Module):
    def __init__(self, word_weights, attribute_weights):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(word_weights.clone(), freeze=False, padding_idx=0)
        self.attribute_embedding = nn.Embedding.from_pretrained(attribute_weights.clone(), freeze=False)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(200, 256, batch_first=True, bidirectional=True)
        self.initial_state = nn.Linear(200, 4 * 256)
        self.image_projection = nn.Linear(2048, 1024)
        self.attribute_attention = nn.Sequential(nn.Linear(200, 100), nn.ReLU(), nn.Linear(100, 1))
        sizes = [1024, 512, 200]
        self.reconstruction = nn.ModuleList([
            nn.ModuleList([GuidedAttention(m, n) for n in sizes]) for m in sizes])
        self.modality_scores = nn.ModuleList([
            nn.Sequential(nn.Linear(size, 512), nn.Tanh(), nn.Linear(512, 1)) for size in sizes])
        self.modality_projection = nn.ModuleList([
            nn.Sequential(nn.Linear(size, 512), nn.Tanh()) for size in sizes])
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.08, a=-0.16, b=0.16)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.01)
        nn.init.xavier_uniform_(self.image_projection.weight)
        for name, parameter in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(parameter)
            else:
                nn.init.zeros_(parameter)
                if 'bias_ih' in name:
                    with torch.no_grad():
                        parameter[256:512].fill_(1.0)

    def forward(self, tokens, images, attributes):
        lengths = tokens.ne(0).sum(dim=1)
        if (lengths == 0).any():
            raise ValueError('Encountered a text sequence containing only padding')
        text_mask = torch.arange(tokens.size(1), device=tokens.device)[None, :] < lengths[:, None]
        attribute_raw = self.attribute_embedding(attributes)
        attribute_weights = self.attribute_attention(attribute_raw).softmax(dim=1)
        attribute_guidance = (attribute_weights * attribute_raw).sum(dim=1)
        h_forward, h_backward, c_forward, c_backward = self.initial_state(attribute_guidance).relu().chunk(4, dim=1)
        embedded = self.dropout(self.word_embedding(tokens))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed,
                                (torch.stack([h_forward, h_backward]),
                                 torch.stack([c_forward, c_backward])))
        text_raw, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=tokens.size(1))
        text_guidance = (text_raw * text_mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(1)
        image_raw = self.image_projection(images).relu()
        raw = [image_raw, text_raw, attribute_raw]
        guidance = [image_raw.mean(dim=1), text_guidance, attribute_guidance]
        reconstructed = []
        for sequence, attentions, mask in zip(raw, self.reconstruction, [None, text_mask, None]):
            weights = torch.stack([attention(sequence, guide, mask)
                                   for attention, guide in zip(attentions, guidance)]).mean(dim=0)
            reconstructed.append((weights * sequence).sum(dim=1))
        scores = torch.cat([score(vector) for score, vector in
                            zip(self.modality_scores, reconstructed)], dim=1).softmax(dim=1)
        projected = torch.stack([projection(vector) for projection, vector in
                                 zip(self.modality_projection, reconstructed)], dim=1)
        fused = (scores.unsqueeze(-1) * projected).sum(dim=1)
        return self.classifier(fused).squeeze(-1)

    def regularization(self):
        return 0.5 * sum(parameter.square().sum() for name, parameter in self.named_parameters()
                         if parameter.requires_grad and name != 'word_embedding.weight')

@torch.no_grad()
def check_padding(model):
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    tokens = torch.tensor([[1, 2, 0], [3, 0, 0]], device=device)
    images = torch.zeros(2, 196, 2048, device=device)
    attributes = torch.zeros(2, 5, dtype=torch.long, device=device)
    padded = torch.cat([tokens, tokens.new_zeros(2, 4)], dim=1)
    torch.testing.assert_close(model(tokens, images, attributes), model(padded, images, attributes))
    mask = tokens.ne(0)
    raw = torch.zeros(2, 3, 512, device=device)
    for attention, size in zip(model.reconstruction[1], [1024, 512, 200]):
        weights = attention(raw, torch.zeros(2, size, device=device), mask).squeeze(-1)
        assert (weights[~mask] == 0).all(), 'Padding received attention weight'
        torch.testing.assert_close(weights.sum(dim=1), torch.ones(2, device=device))
    try:
        model(torch.zeros_like(tokens), images, attributes)
    except ValueError:
        pass
    else:
        raise AssertionError('All-padding text must be rejected')
    model.train(was_training)


## Train and evaluate

In [4]:
def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    predicted = np.asarray(probabilities) > 0.5
    tp = int(((labels == 1) & predicted).sum()); fp = int(((labels == 0) & predicted).sum())
    tn = int(((labels == 0) & ~predicted).sum()); fn = int(((labels == 1) & ~predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': (tp + tn) / len(labels)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    labels, probabilities, tweet_ids = [], [], []
    for tokens, images, attributes, targets, ids in tqdm(loader, desc='Evaluate', leave=False):
        logits = model(tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True),
                       attributes.to(DEVICE, non_blocking=True))
        probabilities.extend(logits.sigmoid().cpu().tolist())
        labels.extend(targets.tolist()); tweet_ids.extend(ids)
    return binary_metrics(labels, probabilities), tweet_ids, labels, probabilities

RUN_ROOT.mkdir(parents=True, exist_ok=True)
all_results = []
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    train_loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
        generator=torch.Generator().manual_seed(seed), persistent_workers=NUM_WORKERS > 0)
    run_dir = RUN_ROOT / f'seed_{seed}'
    run_dir.mkdir(exist_ok=False)
    feature_manifests = [json.loads(path.read_text()) for path in sorted((FEATURE_DIR / 'manifests').glob('*.json'))]
    config = {'model': 'hierarchical_fusion', 'variant': 'finetuned_features_own_attributes_packed',
              'seed': seed, 'attribute_predictions': str(ATTRIBUTE_PREDICTIONS), 'attribute_embedding': 'trainable',
              'text_padding': 'packed_lstm_masked_mean_masked_attention',
              'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS, 'patience': PATIENCE,
              'learning_rate': LEARNING_RATE, 'lr_decay_per_epoch': 0.8, 'l2': L2,
              'sequence_length': SEQ_LEN, 'hidden_size': 256, 'input_dropout': 0.2, 'gradient_clipping': 5.0,
              'positive_weight': positive_weight, 'checkpoint_selection': 'validation positive-class F1',
              'asset_root': str(ASSET_ROOT), 'feature_dir': str(FEATURE_DIR),
              'split_counts': {'train': len(train_data), 'valid': len(valid_data), 'test': len(test_data)},
              'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
              'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
              'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
              'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
              'device': str(DEVICE), 'num_workers': NUM_WORKERS, 'feature_manifests': feature_manifests}
    (run_dir / 'config.json').write_text(json.dumps(config, indent=2) + '\n')
    base_model = HierarchicalFusion(word_weights, attribute_weights).to(DEVICE)
    check_padding(base_model)
    network = nn.DataParallel(base_model) if torch.cuda.device_count() > 1 else base_model
    print(f'Seed {seed} | GPUs: {torch.cuda.device_count()}')
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.Adam(base_model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
    best_f1, stale_epochs, history = -1.0, 0, []
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        network.train(); epoch_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        for tokens, images, attributes, labels, _ in tqdm(train_loader, desc=f'Seed {seed} epoch {epoch}/{EPOCHS}'):
            tokens, images = tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True)
            attributes, labels = attributes.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(network(tokens, images, attributes), labels) + L2 * base_model.regularization()
            if not torch.isfinite(loss): raise FloatingPointError('Non-finite training loss')
            loss.backward(); nn.utils.clip_grad_norm_(base_model.parameters(), 5.0); optimizer.step()
            epoch_loss += loss.item() * len(labels); seen += len(labels)
        valid_metrics, _, _, _ = evaluate(network, valid_loader)
        record = {'epoch': epoch, 'train_loss': epoch_loss / seen, 'validation': valid_metrics,
                  'seconds': time.perf_counter() - epoch_start}
        history.append(record)
        (run_dir / 'history.json').write_text(json.dumps(history, indent=2) + '\n')
        print(json.dumps({'seed': seed, **record}))
        if valid_metrics['f1'] > best_f1:
            best_f1, stale_epochs = valid_metrics['f1'], 0
            torch.save({'model': base_model.state_dict(), 'epoch': epoch,
                        'validation_f1': best_f1, 'seed': seed}, run_dir / 'best.pt.tmp')
            os.replace(run_dir / 'best.pt.tmp', run_dir / 'best.pt')
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= PATIENCE: break
    training_seconds = time.perf_counter() - start
    checkpoint = torch.load(run_dir / 'best.pt', map_location=DEVICE, weights_only=True)
    base_model.load_state_dict(checkpoint['model'])
    test_start = time.perf_counter()
    test_metrics, ids, labels, probabilities = evaluate(network, test_loader)
    result = {'seed': seed, 'selected_epoch': checkpoint['epoch'],
              'best_validation_f1': checkpoint['validation_f1'], 'test': test_metrics,
              'training_seconds': training_seconds, 'test_seconds': time.perf_counter() - test_start}
    (run_dir / 'results.json').write_text(json.dumps(result, indent=2) + '\n')
    with (run_dir / 'test_predictions.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['tweet_id', 'label', 'probability', 'prediction'])
        writer.writerows((tweet_id, int(label), probability, int(probability > 0.5))
                         for tweet_id, label, probability in zip(ids, labels, probabilities))
    all_results.append(result)
    print(json.dumps(result, indent=2))
    del network, base_model, optimizer, scheduler, criterion
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

summary = {metric: {'mean': float(np.mean([r['test'][metric] for r in all_results])),
                    'std': float(np.std([r['test'][metric] for r in all_results], ddof=1))}
           for metric in ('precision', 'recall', 'f1', 'accuracy')}
summary['seeds'] = all_results
(RUN_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))


Seed 42 | GPUs: 2


Seed 42 epoch 1/8:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 42, "epoch": 1, "train_loss": 0.4832589602239298, "validation": {"precision": 0.809322033898305, "recall": 0.7966631908237748, "f1": 0.8029427220178665, "accuracy": 0.8443983402489627}, "seconds": 620.487494905}


Seed 42 epoch 2/8:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

... (66 outputs left out)

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"seed": 44, "epoch": 7, "train_loss": 0.03479343770043278, "validation": {"precision": 0.7872127872127872, "recall": 0.821689259645464, "f1": 0.8040816326530612, "accuracy": 0.8406639004149378}, "seconds": 165.94917350300057}


Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{
  "seed": 44,
  "selected_epoch": 2,
  "best_validation_f1": 0.8044579533941236,
  "test": {
    "precision": 0.7727272727272727,
    "recall": 0.8508863399374348,
    "f1": 0.8099255583126551,
    "accuracy": 0.8410128684101287
  },
  "training_seconds": 1130.0540173090003,
  "test_seconds": 34.921430949999376
}
{
  "precision": {
    "mean": 0.7689056286691454,
    "std": 0.003760635545034114
  },
  "recall": {
    "mean": 0.8373305526590197,
    "std": 0.014709839394854918
  },
  "f1": {
    "mean": 0.8016257050243513,
    "std": 0.007836194094237593
  },
  "accuracy": {
    "mean": 0.8350629583506296,
    "std": 0.00535368538042151
  },
  "seeds": [
    {
      "seed": 42,
      "selected_epoch": 7,
      "best_validation_f1": 0.8120531154239019,
      "test": {
        "precision": 0.7687804878048781,
        "recall": 0.821689259645464,
        "f1": 0.7943548387096774,
        "accuracy": 0.8306351183063512
      },
      "training_seconds": 1667.820970525,
      "test_seconds